# MedCPT Query Encoder: ML Registry + SPCS Deployment

This notebook registers the **MedCPT Query Encoder** (`ncbi/MedCPT-Query-Encoder`) in the Snowflake ML Registry and deploys it as an SPCS service for real-time inference.

**Why a separate Query Encoder?**
MedCPT uses an asymmetric architecture - articles and queries are encoded by different models. The article embeddings are pre-computed in bulk by the Article Encoder. This notebook handles the query side, which runs at search time to encode user questions into the same 768-dim vector space.

**Flow:**
1. Download the HuggingFace model locally
2. Wrap it in a `CustomModel` class with an `encode` inference API
3. Test locally to validate output shape
4. Register in Snowflake ML Registry (`MEDCPT_QUERY_ENCODER`)
5. Deploy as an SPCS service (`medcpt_query_encoder_svc`) on GPU
6. Test end-to-end: encode a query and search Cortex Search via multi-index query
7. Deploy the Article Encoder as a separate SPCS service for incremental embedding loads

In [ ]:
!pip install sentence-transformers

## Setup: Session, Registry, and imports

Connect to Snowflake and initialize the ML Registry. `torch` is needed here because we load the model locally first before registering it.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model
from snowflake.snowpark.context import get_active_session
import pandas as pd
import torch

session = get_active_session()

reg = Registry(
    session=session,
    database_name="SFSE_PUBMED_DB",
    schema_name="PUBMED",
)

session.sql('USE SCHEMA SFSE_PUBMED_DB.PUBMED').collect()

## Define the MedCPT Query Encoder custom model

Download `ncbi/MedCPT-Query-Encoder` from HuggingFace, save to a local directory, and wrap it in a `CustomModel` subclass. The `encode` inference API accepts a DataFrame with a `TEXT` column and returns 768-dim embeddings.

The model is saved locally so it can be bundled as an artifact when registering with the ML Registry.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
MODEL_LOCAL_DIR = "/tmp/medcpt_query_encoder"

local_model = SentenceTransformer(
    MODEL_NAME,
    trust_remote_code=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
local_model.save(MODEL_LOCAL_DIR)

class MedCPTQueryEncoder(custom_model.CustomModel):
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        self.model = SentenceTransformer(
            self.context["model_dir"],
            trust_remote_code=True,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )

    @custom_model.inference_api
    def encode(self, input: pd.DataFrame) -> pd.DataFrame:
        texts = input["TEXT"].tolist()
        embs = self.model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
        return pd.DataFrame({"EMBEDDING": embs.tolist()})

## Local validation

Instantiate the model with its context and run a test encode to verify output shape (should be 768 dimensions).

In [ ]:
mc = custom_model.ModelContext(
    artifacts={"model_dir": MODEL_LOCAL_DIR},
)

medcpt_query = MedCPTQueryEncoder(mc)

sample_input = pd.DataFrame({
    "TEXT": ["aortic stenosis treatment outcomes"]
})

result = medcpt_query.encode(sample_input)
print(f"Output rows: {len(result)}")
print(f"Embedding dim: {len(result['EMBEDDING'].iloc[0])}")
print(f"First 5 values: {result['EMBEDDING'].iloc[0][:5]}")

## Register in Snowflake ML Registry

Log the model to the registry with GPU options enabled. `pip_requirements` ensures `sentence-transformers`, `torch`, and `transformers` are installed in the SPCS container at runtime.

In [ ]:
mv = reg.log_model(
    medcpt_query,
    model_name="MEDCPT_QUERY_ENCODER",
    version_name="v1",
    sample_input_data=sample_input,
    pip_requirements=["sentence-transformers", "torch", "transformers"],
    options={"use_gpu": True, "cuda_version": "12.3"},
)

## Retrieve the registered model version

Fetch the model version reference from the registry (useful for deploying or re-deploying without re-logging).

In [ ]:
mv = reg.get_model(model_name="MEDCPT_QUERY_ENCODER").version("V1")
mv

## Deploy as SPCS service

Create the `medcpt_query_encoder_svc` service on `PUBMED_GPU_S_POOL`. This is a lightweight single-GPU service since query encoding is fast (single query at a time). `max_batch_rows=64` is sufficient for search workloads.

In [ ]:
mv.create_service(
    service_name="medcpt_query_encoder_svc",
    service_compute_pool="PUBMED_GPU_S_POOL",
    ingress_enabled=True,
    gpu_requests="1",
    min_instances=1,
    max_instances=1,
    num_workers=1,
    max_batch_rows=64,
)

## Test: Encode a query via the SPCS service

Call the deployed service directly via SQL to verify it returns embeddings.

In [ ]:
SELECT medcpt_query_encoder_svc!encode('aortic stenosis treatment outcomes') AS result;

## End-to-end test: Query Encoder + Cortex Search multi-index query

Encode a query via the SPCS service, then pass the resulting vector to Cortex Search. The `multi_index_query` parameter sends the user-provided vector to the EMBEDDING vector index while Cortex Search also searches the CHUNK text index. This combines semantic vector similarity with keyword relevance.

In [ ]:

import json
from snowflake.core import Root

query_text='outcomes of aortic stenosis'

safe_query = query_text.replace("'", "''")
result = session.sql(
    f"SELECT medcpt_query_encoder_svc!encode('{safe_query}'):EMBEDDING AS vec"
).collect()
query_vector = json.loads(result[0]["VEC"])



In [ ]:
root = Root(session)
svc = (root
    .databases["SFSE_PUBMED_DB"]
    .schemas["PUBMED"]
    .cortex_search_services["PUBMED_OA_MEDCPT_SEARCH_SVC"]
)

resp = svc.search(
    multi_index_query={
        "EMBEDDING": [{"vector": query_vector}]
    },
    columns=["CHUNK", "ARTICLE_URL", "KEY"],
    limit=5
)
resp.results


In [ ]:
#Deploy article encoder for incremental runs
mv = reg.get_model(model_name="MEDCPT_EMBEDDER").version("V1")
mv.create_service(
    service_name="medcpt_embedder_incremental_svc",
    service_compute_pool="PUBMED_GPU_S_POOL",
    ingress_enabled=True,
    gpu_requests="1",
    min_instances=1,
    max_instances=1,
    max_batch_rows=256,
)

## Deploy Article Encoder for incremental embedding loads

Deploy a separate SPCS service for the MedCPT Article Encoder (`MEDCPT_EMBEDDER`). This service is used by the daily incremental load task to generate embeddings for new/updated PubMed articles. It runs on `PUBMED_GPU_S_POOL` with `max_batch_rows=256` for bulk throughput.

This service is managed by the `LOAD_PUBMED_INCREMENTAL` stored procedure, which resumes it before the load and suspends it after.